# Every operation, one at a time

One seed passage and all eighteen operations, each with a section of its own:
what it means, why it is in the inventory at the cost it has, and how it is
actually implemented.

Every step is written out in full rather than hidden behind a helper, so the
mechanics stay visible — each cell rebuilds the reuse before and after, and
prints the `EditOperation` that was just recorded.

At the end the accumulated script is printed, checked for structural validity,
replayed against the seed, and inverted back to it.

This is the **generation** direction: an operation is *told* what to do. The
detection direction — inferring from a pair which operation applies — is much
harder, and six of the eighteen cannot do it at all. Each section says which.

In [1]:
import sys
sys.path.insert(0, ".")

from retexo import OperationRegistry, Scriba, VariantBuilder

operation_registry = OperationRegistry.default()
script_verifier = Scriba(operation_registry)

operation_registry.to_frame()

,level,role,cost,detectable,requires
tag,,,,,
QUOTE,span,writing,0.00,True,—
FRAME,span,writing,0.50,False,citation_verbs
ADAPT,span,marker,0.00,False,stopwords
DISPERSE,span,marker,1.50,True,clauses
NOP,token,writing,0.00,True,—
MORPH,token,writing,0.25,True,morphology
POS,token,writing,0.50,False,derivation
NE-SUB,token,writing,0.75,False,entities
SYN,token,writing,0.50,True,wordnet


## The seed passage

*Aeneid* 1.1–3, trimmed to seventeen tokens — chosen so that every operation
has somewhere to land and no source token is left unaccounted for. The numbers
are the positions the operations refer to.

In [2]:
SEED_PASSAGE = (
    "arma uirumque cano Troiae qui primus ab oris Italiam "
    "fato profugus Lauiniaque uenit litora multum et terris"
)
seed_passage_tokens = SEED_PASSAGE.split()

for index, token in enumerate(seed_passage_tokens):
    print(f"{index:>2}  {token}")
print(f"\n{len(seed_passage_tokens)} tokens")

 0  arma
 1  uirumque
 2  cano
 3  Troiae
 4  qui
 5  primus
 6  ab
 7  oris
 8  Italiam
 9  fato
10  profugus
11  Lauiniaque
12  uenit
13  litora
14  multum
15  et
16  terris

17 tokens


## The builder

`VariantBuilder` records operations without applying them. `build()` replays
everything recorded so far through Scriba and returns the reuse plus the
script, and it is non-destructive — so calling it after each step shows the
passage growing.

Reuse positions are handed out in the order operations are applied, so each
writing operation appends to the right. Operations that write nothing leave the
passage unchanged, which is exactly the point some of them make.

In [3]:
reuse_builder = VariantBuilder(SEED_PASSAGE, operation_registry)

print("source tokens :", len(reuse_builder.source_tokens))
print("recorded so far:", len(reuse_builder.build()[1].operations))

source tokens : 17
recorded so far: 0


## 1 · `FRAME` — span level

The citing author's own words wrapped around a quotation — *ut ait Maro*, "as
Maro says". It is the prose that introduces a quotation rather than any part of
the quotation itself.

It is priced **once**, at 0.50, rather than as one insertion per word, because
introducing a quotation is a single authorial act. This matters more than it
sounds: when Ambrose wraps twelve words of Virgil in eighty words of his own,
charging that as eighty insertions would swamp the actual reuse and teach a
model that reuse is mostly insertion. Getting this wrong was the largest single
error in the first version of the pipeline.

**How it is implemented.** `Frame` in `operations/span.py`. Role is *writing*: it occupies reuse positions
but consumes no source, so `source_indices` stays empty. Its inverse is itself
— running the script backwards turns a written frame into a consumed one.

`detectable = False`, and `requires = ("citation_verbs",)`. Generating a frame
is trivial; *detecting* one needs a rule for where the frame stops and the
quotation begins, which is not settled, so the oracle never emits it. That is
why FRAME appears only in synthetic data.

In [4]:
reuse_before, _ = reuse_builder.build()

reuse_builder.frame(["ut", "ait", "Maro"], detail="attribution formula")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : (nothing written yet)
after    : ut ait Maro
written  : ut ait Maro

recorded : FRAME | source () () -> target (0, 1, 2) ('ut', 'ait', 'Maro')
cost     : 0.5


## 2 · `QUOTE` — span level

A contiguous run carried over verbatim, at least four tokens long. It costs
**nothing**: copying is not distance.

It earns a place in the inventory by encoding *contiguity*, which the token
level cannot express. Four consecutive shared words and four scattered ones
both produce four zero-cost copies and are indistinguishable in the operation
profile — yet only the first is a quotation.

**How it is implemented.** `Quote` in `operations/span.py`. Role *writing*, its own inverse. The builder's
`quote()` copies the source tokens straight through to the target, so
`source_tokens` and `target_tokens` are identical.

It is `detectable = True` and needs no resources — contiguity and identity are
readable off the strings alone. The four-token minimum is documented rather
than enforced in code.

In [5]:
reuse_before, _ = reuse_builder.build()

reuse_builder.quote([0, 1, 2, 3], detail="verbatim core, 4 tokens")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : ut ait Maro
after    : ut ait Maro arma uirumque cano Troiae
written  : arma uirumque cano Troiae

recorded : QUOTE | source (0, 1, 2, 3) ('arma', 'uirumque', 'cano', 'Troiae') -> target (3, 4, 5, 6) ('arma', 'uirumque', 'cano', 'Troiae')
cost     : 0.0


## 3 · `NOP` — token level

The token is identical after normalization. Free, and by far the most common
operation in near-verbatim reuse.

Its frequency is the central difficulty of the whole task. When most tokens are
copies, a model that predicts "copy" everywhere scores well while explaining
nothing — the same collapse Reid & Neubig hit and responded to by changing
their output structure rather than reweighting the loss.

**How it is implemented.** `Nop` in `operations/token.py`, subclassing `_TokenOperation`. Its own inverse.

It is one of the few operations with a real `detect()`: it normalizes both
tokens and compares them. Normalization handles Latin orthographic variation —
u/v, i/j, and folds like *th*/*t* and *ph*/*p* — so *uox* and *vox* count as
identical.

In [6]:
reuse_before, _ = reuse_builder.build()

reuse_builder.keep(4, detail="carried over untouched")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : ut ait Maro arma uirumque cano Troiae
after    : ut ait Maro arma uirumque cano Troiae qui
written  : qui

recorded : NOP | source (4,) ('qui',) -> target (7,) ('qui',)
cost     : 0.0


## 4 · `MORPH` — token level

Same lemma, different inflection: *primus* becomes *primos*. Cheap at 0.25,
because the word is the same word — only its grammatical form moved.

In a heavily inflected language this is the workhorse. A quotation absorbed
into a new sentence almost always has to change case, number or tense to fit
its new syntax, and none of that is a change of meaning.

**How it is implemented.** `Morph` in `operations/token.py`. Its own inverse. `requires =
("morphology",)`.

`detect()` asks the morphology resource for both lemmas and fires when they
match, returning `lemma=<the lemma>` as its justification. This is one of the
better-served operations for Latin — CLTK, LatinCy and Collatinus all provide
lemmatization — so coverage here is not the bottleneck.

In [7]:
reuse_before, _ = reuse_builder.build()

reuse_builder.inflect(5, "primos", detail="nom sg -> acc pl")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : ut ait Maro arma uirumque cano Troiae qui
after    : ut ait Maro arma uirumque cano Troiae qui primos
written  : primos

recorded : MORPH | source (5,) ('primus',) -> target (8,) ('primos',)
cost     : 0.25


## 5 · `DEL` — structural level

A source token the reuse does not carry over. Expensive, at 1.00.

`INS` and `DEL` are the two most expensive operations on purpose. **Any** pair
of passages can be related by deleting the whole source and inserting the whole
reuse, so if that route were cheap the distance would collapse and every pair
would score alike. That degenerate script is the baseline every result in the
paper has to be measured against.

**How it is implemented.** `Del` in `operations/structural.py`. Role is *consuming* — the only operation
with that role. It occupies a source position and writes nothing, so
`target_indices` stays empty and the reuse does not grow. Its inverse is `INS`.

Detectable with no resources: after alignment, any source token nothing points
at is a deletion. In the token-classifier model it is never predicted directly
but **derived** from the pointer pattern, which denies the model the shortcut
of explaining a hard pair as pure deletion and insertion.

In [8]:
reuse_before, _ = reuse_builder.build()

reuse_builder.delete(6, detail="preposition dropped")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : ut ait Maro arma uirumque cano Troiae qui primos
after    : ut ait Maro arma uirumque cano Troiae qui primos
written  : nothing

recorded : DEL | source (6,) ('ab',) -> target () ()
cost     : 1.0


## 6 · `HYPER` — token level

Generalization: the reuse names a broader category than the source. *oris*
(shores) becomes *locis* (places).

At 0.75 it costs more than a synonym, because moving up a taxonomy loses
information in a way that swapping equivalents does not.

**How it is implemented.** `Hyper` in `operations/token.py`, subclassing `_WordNetOperation` with
`relation = "hypernyms"`. Its inverse is `HYPO` — generalising and specifying
are the same move in opposite directions, and they are the only tags in the
inventory that are not their own inverse.

`detect()` looks the source lemma up in Latin WordNet and checks whether the
target lemma appears in the hypernym set. It then applies a **vector veto**: if
distributional similarity falls below `ATTESTED_SIMILARITY_FLOOR` (0.35), the
link is rejected even though the wordnet attests it. A pair with no vectors is
accepted on the wordnet's word alone, so the veto never costs coverage where it
cannot judge.

In [9]:
reuse_before, _ = reuse_builder.build()

reuse_builder.hyper(7, "locis", detail="shores -> places")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : ut ait Maro arma uirumque cano Troiae qui primos
after    : ut ait Maro arma uirumque cano Troiae qui primos locis
written  : locis

recorded : HYPER | source (7,) ('oris',) -> target (9,) ('locis',)
cost     : 0.75


## 7 · `NE-SUB` — token level

One named entity substituted for another: *Italiam* becomes *Hesperiam*,
another name for Italy.

It is separated from plain synonymy at 0.75 because swapping a name is a
different authorial act from swapping a common noun — often a deliberate
literary gesture, reaching for the archaic or poetic name.

**How it is implemented.** `NeSub` in `operations/token.py`. Its own inverse. `requires =
("entities",)`.

`detectable = False`. Generation is trivial, detection is not: deciding that
two proper nouns stand in for one another needs a named-entity pass. The
capitalization test an earlier prototype used fires on any sentence-initial
word, because capitalization in Latin critical editions is the modern editor's
choice, not the author's.

In [10]:
reuse_before, _ = reuse_builder.build()

reuse_builder.ne_sub(8, "Hesperiam", detail="Italia -> Hesperia")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : ut ait Maro arma uirumque cano Troiae qui primos locis
after    : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam
written  : Hesperiam

recorded : NE-SUB | source (8,) ('Italiam',) -> target (10,) ('Hesperiam',)
cost     : 0.75


## 8 · `SYN` — token level

An attested synonym — the two words share a WordNet synset. *fato* (by fate)
becomes *sorte* (by lot). Costs 0.50.

This is the operation a lexical resource can actually vouch for, and it is the
one whose coverage limits everything downstream.

**How it is implemented.** `Syn` in `operations/token.py`, a `_WordNetOperation` with `relation =
"synonyms"`. Its own inverse. Same vector veto as `HYPER`.

**This is where the pipeline's biggest weakness lives.** Latin WordNet covers
roughly 30% of lemmas, so most genuine synonym substitutions in real pairs
cannot be attested — and everything the oracle cannot attest falls through to
INS plus DEL. That is why about 81% of teacher-derived labels are INS/DEL.
Moritz et al. wrote down this exact three-way limitation — lemmatiser coverage,
wordnet coverage, and existence of the relation — back in 2016.

In [11]:
reuse_before, _ = reuse_builder.build()

reuse_builder.syn(9, "sorte", detail="wordnet synset")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam
after    : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte
written  : sorte

recorded : SYN | source (9,) ('fato',) -> target (11,) ('sorte',)
cost     : 0.5


## 9 · `POS` — token level

A derivational shift: same root, different word class. *profugus* (adjective,
exiled) becomes *profugium* (noun, refuge). The idea survives the change of
part of speech.

**How it is implemented.** `Pos` in `operations/token.py`. Its own inverse. `requires =
("derivation",)`.

`detectable = False`, and for an instructive reason: a true derivational pair
**crosses lemmas** — *amāre* to *amor* — so lemmatization cannot find it by
construction, and no derivational lexicon for Latin has been identified for the
pipeline. LiLa's Lemma Bank models word formation and would be the place to
look.

In [12]:
reuse_before, _ = reuse_builder.build()

reuse_builder.pos(10, "profugium", detail="adjective -> noun")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte
after    : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium
written  : profugium

recorded : POS | source (10,) ('profugus',) -> target (12,) ('profugium',)
cost     : 0.5


## 10 · `SPLIT` — cardinality level

One source token expands into several contiguous reuse tokens. Here the Latin
enclitic in *Lauiniaque* separates into *Lauinia* and *que*.

Latin reuse does this constantly: a participle unpacked into a relative clause,
the perfect passive written analytically. Without this operation every such
case is described as a deletion followed by insertions, which both inflates the
residual and misnames what the author did.

**How it is implemented.** `Split` in `operations/cardinality.py`. Its inverse is `MERGE`. `requires =
("morphology",)`.

The builder allocates one target index per new token, so a single
`source_indices` entry maps to several `target_indices` — this is the first
operation where the counts stop matching one-to-one.

`detectable = False`. Recognising that a reuse span is the analytic expansion
of one source word needs a periphrasis model beyond lemmatization; only the
perfect passive is special-caseable. These are also the operations that force
the aligner beyond one-to-one matching, which is why they stay switched off
until a diagnostic shows how much residual they would recover.

In [13]:
reuse_before, _ = reuse_builder.build()

reuse_builder.split(11, ["Lauinia", "que"], detail="enclitic separated")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium
after    : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que
written  : Lauinia que

recorded : SPLIT | source (11,) ('Lauiniaque',) -> target (13, 14) ('Lauinia', 'que')
cost     : 0.75


## 11 · `SYN-DIST` — token level

A distributional near-synonym: no WordNet edge, but the two words sit close in
vector space. *uenit* (came) becomes *tetigit* (reached).

This is the operation that catches what the dictionary misses — and it costs
**more** than an attested synonym (0.60 against 0.50) on purpose. The cost
measures how much the link is *believed*, not only how far the meaning moved.

**How it is implemented.** `SynDist` in `operations/token.py`. Its own inverse. `requires = ("vectors",)`.

`detect()` fires when cosine similarity clears `SYN_DIST_THRESHOLD`, which is
**0.65 and calibrated rather than asserted**: sweeping thresholds against
in-domain negatives showed precision rising slowly to 0.68 at 0.60, then
stepping to 0.81 at 0.65, against a chance baseline of 0.47. That step is where
the measure stops describing co-occurrence and starts describing similarity.

Chosen for precision deliberately — a false SYN-DIST invents a relation that is
not there, whereas a missed one costs an insertion and a deletion, which the
residual honestly reports.

In [14]:
reuse_before, _ = reuse_builder.build()

reuse_builder.syn_dist(12, "tetigit", detail="cos = 0.71")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que
after    : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit
written  : tetigit

recorded : SYN-DIST | source (12,) ('uenit',) -> target (15,) ('tetigit',)
cost     : 0.6


## 12 · `HYPO` — token level

Specification, the exact inverse of `HYPER`: the reuse names something narrower
than the source. *litora* (shores) becomes *harenas* (sands). Also 0.75.

**How it is implemented.** `Hypo` in `operations/token.py`, a `_WordNetOperation` with `relation =
"hyponyms"`. `inverse_tag()` returns `HYPER`.

Note what this means for the script as a whole: because HYPER and HYPO swap
under inversion, running a script backwards is not merely relabelling — the
direction of the semantic move is part of what the script records.

In [15]:
reuse_before, _ = reuse_builder.build()

reuse_builder.hypo(13, "harenas", detail="shores -> sands")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit
after    : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit harenas
written  : harenas

recorded : HYPO | source (13,) ('litora',) -> target (16,) ('harenas',)
cost     : 0.75


## 13 · `ANT` — token level

An antonym, usually alongside a negation. *multum* (much) becomes *parum*
(little).

At 1.00 it is the most expensive token operation — as costly as an insertion.
Reversing the sense is the largest single-word change an author can make while
still visibly reusing the source.

**How it is implemented.** `Ant` in `operations/token.py`, a `_WordNetOperation` with `relation =
"antonyms"`. Its own inverse, and detectable where the wordnet has the edge.

Antonymy is sparse in Latin WordNet, so in practice this fires rarely — in the
benchmark's real pairs it is among the rarest tags, which is part of why the
macro-averaged score is hard to move.

In [16]:
reuse_before, _ = reuse_builder.build()

reuse_builder.ant(14, "parum", detail="much -> little")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit harenas
after    : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit harenas parum
written  : parum

recorded : ANT | source (14,) ('multum',) -> target (17,) ('parum',)
cost     : 1.0


## 14 · `MERGE` — cardinality level

The inverse of `SPLIT`: several contiguous source tokens compress into one.
Here *et terris* becomes *terrisque*, folding the conjunction into an enclitic —
which is genuinely how Latin coordination works.

**How it is implemented.** `Merge` in `operations/cardinality.py`. Inverse is `SPLIT`. Same `requires` and
the same `detectable = False`, for the same periphrasis reason.

The builder maps several `source_indices` onto a single `target_indices` entry,
the mirror image of `SPLIT`.

In [17]:
reuse_before, _ = reuse_builder.build()

reuse_builder.merge([15, 16], "terrisque", detail="conjunction becomes enclitic")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit harenas parum
after    : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit harenas parum terrisque
written  : terrisque

recorded : MERGE | source (15, 16) ('et', 'terris') -> target (18,) ('terrisque',)
cost     : 0.75


## 15 · `INS` — structural level

A reuse token with no source at all. The connective *tandem*, coming from
nowhere in the seed. Costs 1.00.

Together with `DEL` this is the pair that can explain *any* two passages, which
is why a script made only of them is the degenerate baseline. On the held-out
evaluation slice that baseline scores 0.661 typed F1 — and the trained model
scores 0.647, which is the honest headline number for the current results.

**How it is implemented.** `Ins` in `operations/structural.py`. Role *writing*, inverse `DEL`. It
consumes no source, so `source_indices` is empty.

Like `DEL`, it is never predicted directly by the token classifier but derived:
a reuse position whose pointer selects the null source *is* an insertion.
Deriving both keeps the two statements consistent by construction.

In [18]:
reuse_before, _ = reuse_builder.build()

reuse_builder.insert("tandem", detail="connective with no source")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit harenas parum terrisque
after    : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit harenas parum terrisque tandem
written  : tandem

recorded : INS | source () () -> target (19,) ('tandem',)
cost     : 1.0


## 16 · `REORDER` — structural level — marker

An aligned pair whose position crosses another's. Costs 0.50.

This is the first **marker**: it writes no token and consumes no source
position. The movement itself is already encoded in the index mapping of the
operations that write the tokens, so emitting positions here would cover them
twice.

**How it is implemented.** `Reorder` in `operations/structural.py`, role *marker*. Its own inverse.

It exists purely **to be counted**. The distance has to be recoverable from the
operation profile alone — a linear model over the counts — so folding
reordering's cost into a neighbouring operation would silently break that
property. The last section of this notebook checks the linearity that this
choice protects.

In the script it appears with `source_indices` filled but `target_indices`
empty, which is how a marker is represented.

In [19]:
reuse_before, _ = reuse_builder.build()

reuse_builder.mark_reordered(4, detail="crosses a neighbour")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit harenas parum terrisque tandem
after    : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit harenas parum terrisque tandem
written  : nothing

recorded : REORDER | source (4,) ('qui',) -> target () ()
cost     : 0.5


## 17 · `ADAPT` — span level — marker

A region reused with its substitutions concentrated inside it — the counterpart
of `QUOTE` for non-verbatim reuse. Here positions 7 to 10, where the HYPER,
NE-SUB, SYN and POS steps clustered.

It costs **nothing**. It is a description of *where* the reworking happened,
not a claim that reworking is itself extra distance — the token operations
inside it are already priced.

**How it is implemented.** `Adapt` in `operations/span.py`, role *marker*. Its own inverse. `requires =
("stopwords",)`.

Because it is a marker rather than a covering span, the token operations inside
the region still stand; it annotates them rather than replacing them. That
distinction is what keeps every reuse position accounted for exactly once, and
so keeps the cost linear.

`detectable = False`: identifying a concentration of substitutions needs a
stoplist to tell content words from function words.

In [20]:
reuse_before, _ = reuse_builder.build()

reuse_builder.mark_adapted([7, 8, 9, 10], detail="substitutions cluster here")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit harenas parum terrisque tandem
after    : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit harenas parum terrisque tandem
written  : nothing

recorded : ADAPT | source (7, 8, 9, 10) ('oris', 'Italiam', 'fato', 'profugus') -> target () ()
cost     : 0.0


## 18 · `DISPERSE` — span level — marker

Shared material split across a clause boundary. At 1.50 it is the most
expensive operation in the whole inventory.

**It is the only operation that is evidence *against* reuse.** Shared words
straddling a clause boundary are usually coincidence, so it is priced above
insertion and deletion specifically to push a pair apart rather than draw it
together.

**How it is implemented.** `Disperse` in `operations/span.py`, role *marker*. Its own inverse. `requires =
("clauses",)`, and it is `detectable = True`.

One honest caveat is recorded in the source: clause boundaries in Latin
critical editions are supplied by **modern editors**, so this operation
measures the edition as much as it measures the text.

In [21]:
reuse_before, _ = reuse_builder.build()

reuse_builder.mark_dispersed([15, 16], detail="straddles a clause break")

reuse_after, script_so_far = reuse_builder.build()
recorded = script_so_far.operations[-1]

print("before   :", " ".join(reuse_before) or "(nothing written yet)")
print("after    :", " ".join(reuse_after))
print("written  :", " ".join(reuse_after[len(reuse_before):]) or "nothing")
print()
print("recorded :", recorded.tag,
      "| source", recorded.source_indices, recorded.source_tokens,
      "-> target", recorded.target_indices, recorded.target_tokens)
print("cost     :", operation_registry[recorded.tag].default_cost)

before   : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit harenas parum terrisque tandem
after    : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit harenas parum terrisque tandem
written  : nothing

recorded : DISPERSE | source (15, 16) ('et', 'terris') -> target () ()
cost     : 1.5


# The finished script

Eighteen operations, each used exactly once. The script is the complete account
of how the seed became the reuse — and it is the object the model is trained to
produce.

In [22]:
reuse_passage, edit_script = reuse_builder.build()

print("seed passage  :", SEED_PASSAGE)
print()
print("reuse passage :", " ".join(reuse_passage))
print()
print(edit_script.human())

seed passage  : arma uirumque cano Troiae qui primus ab oris Italiam fato profugus Lauiniaque uenit litora multum et terris

reuse passage : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit harenas parum terrisque tandem

FRAME                            — -> ut ait Maro               [attribution formula]
QUOTE     arma uirumque cano Troiae -> arma uirumque cano Troiae  [verbatim core, 4 tokens]
NOP                            qui -> qui                       [carried over untouched]
MORPH                       primus -> primos                    [nom sg -> acc pl]
DEL                             ab -> —                         [preposition dropped]
HYPER                         oris -> locis                     [shores -> places]
NE-SUB                     Italiam -> Hesperiam                 [Italia -> Hesperia]
SYN                           fato -> sorte                     [wordnet synset]
POS                       profugus -> pro

## The same script, serialized

The form a sequence-to-sequence model would have to generate. Every number here
is a position the model must get right and keep internally consistent — which
is the argument for the tagging formulation, where position is read off the
input rather than written into the output.

In [23]:
print(edit_script.serialize())

FRAME ->0,1,2 ut ait Maro | QUOTE 0,1,2,3>3,4,5,6 arma uirumque cano Troiae | NOP 4>7 qui | MORPH 5>8 primos | DEL 6>- | HYPER 7>9 locis | NE-SUB 8>10 Hesperiam | SYN 9>11 sorte | POS 10>12 profugium | SPLIT 11>13,14 Lauinia que | SYN-DIST 12>15 tetigit | HYPO 13>16 harenas | ANT 14>17 parum | MERGE 15,16>18 terrisque | INS ->19 tandem | REORDER 4>- | ADAPT 7,8,9,10>- | DISPERSE 15,16>-


## Validate and replay

Two separate questions, reported separately because they are different
failures. **Validity** is structural — does the script write a reuse position
twice, consume a source token twice, leave a gap? **Verification** runs it:
replay on the seed and check the result is the reuse.

Scriba shares no alignment code with the builder. That independence is the
whole point: a checker built from the machinery that produced a script could
only ever re-confirm it, whereas Scriba can judge a script from any source,
including a model's.

In [24]:
structural_validity = script_verifier.validate(edit_script)
print("structurally valid :", bool(structural_validity))
for problem in structural_validity.problems:
    print("  -", problem)

replayed_tokens = script_verifier.execute(edit_script, seed_passage_tokens)
print()
print("replayed      :", " ".join(replayed_tokens))
print("reuse passage :", " ".join(reuse_passage))
print()
print("replay reproduces the reuse :",
      script_verifier.verify(edit_script, seed_passage_tokens, reuse_passage))

structurally valid : True

replayed      : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit harenas parum terrisque tandem
reuse passage : ut ait Maro arma uirumque cano Troiae qui primos locis Hesperiam sorte profugium Lauinia que tetigit harenas parum terrisque tandem

replay reproduces the reuse : True


## Invert

Every operation records both what it consumed and what it wrote, so the script
runs backwards. Most tags are their own inverse; the exceptions are the three
pairs `INS`/`DEL`, `SPLIT`/`MERGE` and `HYPER`/`HYPO`.

Invertibility is what makes the script a *distance* rather than a one-way
description: the cost of getting from the seed to the reuse equals the cost of
getting back.

One thing to read carefully below — the `detail` gloss is carried over from the
forward direction unchanged. The inverted `HYPO` step still reads
`shores -> places`, describing the original move rather than the one being
replayed. The tag flips; the human-written note does not.

In [25]:
inverse_script = edit_script.invert()
print(inverse_script.human())

FRAME                  ut ait Maro -> —                         [attribution formula]
QUOTE     arma uirumque cano Troiae -> arma uirumque cano Troiae  [verbatim core, 4 tokens]
NOP                            qui -> qui                       [carried over untouched]
MORPH                       primos -> primus                    [nom sg -> acc pl]
INS                              — -> ab                        [preposition dropped]
HYPO                         locis -> oris                      [shores -> places]
NE-SUB                   Hesperiam -> Italiam                   [Italia -> Hesperia]
SYN                          sorte -> fato                      [wordnet synset]
POS                      profugium -> profugus                  [adjective -> noun]
MERGE                  Lauinia que -> Lauiniaque                [enclitic separated]
SYN-DIST                   tetigit -> uenit                     [cos = 0.71]
HYPER                      harenas -> litora                    [shor

In [26]:
restored_tokens = script_verifier.execute(inverse_script, reuse_passage)

print("restored     :", " ".join(restored_tokens))
print("seed passage :", SEED_PASSAGE)
print()
print("returns to the seed :",
      script_verifier.verify(inverse_script, reuse_passage, seed_passage_tokens))
print("cost is unchanged   :",
      abs(inverse_script.cost() - edit_script.cost()) < 1e-9)

restored     : arma uirumque cano Troiae qui primus ab oris Italiam fato profugus Lauiniaque uenit litora multum et terris
seed passage : arma uirumque cano Troiae qui primus ab oris Italiam fato profugus Lauiniaque uenit litora multum et terris

returns to the seed : True
cost is unchanged   : True
